# Drive Pulse — ML Training Pipeline

This notebook walks through the full ML lifecycle: dataset building, training, benchmarking and model promotion. It calls the **same production scripts** the backend uses, so every number here matches what runs in the app.

> Run cells top-to-bottom. Each step prints its own metrics.

## 0. Setup

The pipeline reads labeled trips from the database, computes 16 vehicle-aware features per trip, and trains a risk classifier (safe vs risky). Label priority: human admin reviews → external benchmark labels (Kaggle) → synthetic ground truth → weak rules.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from scripts import build_training_dataset as btd
from scripts import train_model_v2 as tmv2
from scripts import benchmark_models as bm

print('Modules loaded — production scripts ready.')

## 1. Build the training dataset

Every completed trip is re-scored through the shared pipeline (preprocessing → features → rules → vehicle-aware thresholds), then assigned a label from the priority tiers. The output CSV is the single source for training.

In [ ]:
summary = btd.main()
print(json.dumps(summary, indent=2, default=str)[:2000])

## 2. Train the production candidates

Two candidates are trained with **stratified 5-fold cross-validation** (honest, split-independent metrics): a balanced Logistic Regression and a tuned Gradient Boosting classifier. The decision threshold is tuned out-of-fold to maximise risky-trip F1 while keeping the false-positive rate within the promotion gate.

In [ ]:
report = tmv2.train_with_cv()
if report:
    print('Best model:', report['best_model_version'])
    print('OOF metrics:', json.dumps(report['best_oof_metrics'], indent=2))
    print('Reaches 80% risky-F1 target:', report['best_reaches_risky_f1_target'])

## 3. Benchmark against the wider field

The production candidates are compared against Random Forest, SVM, k-NN, Naive Bayes and Decision Tree on the *same* splits with the same threshold tuning — a head-to-head, not a cherry-pick.

In [ ]:
bm.main()

## 4. Feature importance

Which signals actually drive the risk reading? (Tree-based models expose importances directly.)

In [ ]:
import pandas as pd
from app.ml.schemas import FEATURE_COLUMNS_FV1
df = pd.read_csv('artifacts/datasets/trip_features_fv1.csv')
print('Dataset rows:', len(df), '| classes:', df['label_binary'].value_counts().to_dict())
print('\nFeature columns used by the model:')
print('\n'.join(f'  - {c}' for c in FEATURE_COLUMNS_FV1))

## 5. Label sources

Transparency: where did each training label come from? `reviewed_real` = human admin reviews, `reviewed_external` = the Kaggle benchmark dataset, `reviewed_demo` = score-derived demo labels (no independent signal), `weak_label` = rule-score heuristics.

In [ ]:
print(df.groupby(['label_tier','label_source']).size().to_string())

## 6. Promotion

Promotion is gated on calibration metrics (Brier ≤ 0.25, risky-F1 ≥ 0.55, FPR ≤ 0.35). Run the promotion script to move the best candidate into production if it clears the gate.

In [ ]:
# from scripts import promote_model
# promote_model.main()
print('Promotion gate: Brier <= 0.25, risky-F1 >= 0.55, FPR <= 0.35')

---
**Why .py and not a notebook for production?** The pipeline must run headless in the auto-retrain loop, be unit-tested, and be importable by the API. Notebooks keep state in the kernel and can't be imported or CI-tested. This notebook is the presentation layer — it calls the same functions, so it can never drift from production.